In [ ]:
!pip install transformers textblob google-genai --quiet

In [ ]:
import json
from google import genai
from transformers import pipeline
from textblob import TextBlob

# Configure Gemini API
GOOGLE_API_KEY = "actual-key-hidden-for-github"
client = genai.Client(api_key=GOOGLE_API_KEY)

print("Libraries loaded successfully")

Libraries loaded successfully


In [ ]:
# ESG operational messages for classification
esg_messages = [
    "There is a water leak in Building C that has been running all morning.",
    "The recycling bins are contaminated again and no one seems to be checking them.",
    "The air conditioning is running overnight in an empty office.",
    "I want to report that one of our suppliers may not meet our sustainability policy.",
    "The accessible entrance near the main building has been blocked for two days."
]

print(f"Loaded {len(esg_messages)} ESG messages for classification")
for i, msg in enumerate(esg_messages):
    print(f"Message {i+1}: {msg}")

Loaded 5 ESG messages for classification
Message 1: There is a water leak in Building C that has been running all morning.
Message 2: The recycling bins are contaminated again and no one seems to be checking them.
Message 3: The air conditioning is running overnight in an empty office.
Message 4: I want to report that one of our suppliers may not meet our sustainability policy.
Message 5: The accessible entrance near the main building has been blocked for two days.


In [ ]:
def create_esg_prompt(message):
    return f"""
You are an ESG operations analyst for a large organisation.
Your task is to classify and triage sustainability-related
operational messages submitted by employees.

Use the following definitions:
- issue_category: ENERGY, WATER, WASTE, SUPPLIER, ACCESSIBILITY, GOVERNANCE, OTHER
- urgency: CRITICAL, HIGH, MEDIUM, LOW
- sentiment: POSITIVE, NEUTRAL, NEGATIVE
- followup_required: Y or N
- recommended_team: facilities, sustainability, procurement, HR, safety, or management
- escalation_reason: brief reason if HIGH or CRITICAL, otherwise null
- data_sensitivity_risk: HIGH, MEDIUM, LOW
- brief_summary: one sentence summary

Analyse the message and return ONLY valid JSON. No extra text.

Message: "{message}"

Return JSON only:
"""

print("Prompt template defined")

Prompt template defined


In [ ]:
gemini_results = []

for i, message in enumerate(esg_messages):
    print(f"\nProcessing message {i+1}...")
    print(f"Message: {message}")

    prompt = create_esg_prompt(message)

    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )
        raw_text = response.text.strip()
        clean_text = raw_text.replace("```json", "").replace("```", "").strip()
        parsed = json.loads(clean_text)
        gemini_results.append({
            "message": message,
            "classification": parsed
        })
        print(f"Result: {json.dumps(parsed, indent=2)}")

    except Exception as e:
        print(f"Error: {e}")
        gemini_results.append({
            "message": message,
            "classification": {"error": str(e)}
        })

print("\nGemini classification complete!")


Processing message 1...
Message: There is a water leak in Building C that has been running all morning.
Result: {
  "issue_category": "WATER",
  "urgency": "HIGH",
  "sentiment": "NEUTRAL",
  "followup_required": "Y",
  "recommended_team": "facilities",
  "escalation_reason": "Ongoing significant resource waste (water) for an extended period in Building C.",
  "data_sensitivity_risk": "LOW",
  "brief_summary": "Employee reports a continuous water leak in Building C that has been running all morning."
}

Processing message 2...
Message: The recycling bins are contaminated again and no one seems to be checking them.
Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

Processing message 3...
Message: The air conditioning is running overnight in an empty office.
Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is curre

In [ ]:
#cell 5
import time

gemini_results = []

for i, message in enumerate(esg_messages):
    print(f"\nProcessing message {i+1}...")
    print(f"Message: {message}")

    prompt = create_esg_prompt(message)

    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt
            )
            raw_text = response.text.strip()
            clean_text = raw_text.replace("```json", "").replace("```", "").strip()
            parsed = json.loads(clean_text)
            gemini_results.append({
                "message": message,
                "classification": parsed
            })
            print(f"Result: {json.dumps(parsed, indent=2)}")
            break

        except Exception as e:
            if attempt < max_retries - 1:
                print(f"Attempt {attempt+1} failed, retrying in 15 seconds...")
                time.sleep(15)
            else:
                print(f"Error after {max_retries} attempts: {e}")
                gemini_results.append({
                    "message": message,
                    "classification": {"error": str(e)}
                })

    # Wait between messages to avoid rate limits
    time.sleep(5)

print("\nGemini classification complete!")


Processing message 1...
Message: There is a water leak in Building C that has been running all morning.
Attempt 1 failed, retrying in 15 seconds...
Result: {
  "issue_category": "WATER",
  "urgency": "HIGH",
  "sentiment": "NEUTRAL",
  "followup_required": "Y",
  "recommended_team": "facilities",
  "escalation_reason": "Ongoing water waste and potential property damage.",
  "data_sensitivity_risk": "LOW",
  "brief_summary": "An ongoing water leak has been reported in Building C since the morning."
}

Processing message 2...
Message: The recycling bins are contaminated again and no one seems to be checking them.
Attempt 1 failed, retrying in 15 seconds...
Attempt 2 failed, retrying in 15 seconds...
Result: {
  "issue_category": "WASTE",
  "urgency": "MEDIUM",
  "sentiment": "NEGATIVE",
  "followup_required": "Y",
  "recommended_team": "facilities",
  "escalation_reason": null,
  "data_sensitivity_risk": "LOW",
  "brief_summary": "An employee reports recurring contamination of recycling

In [15]:
from transformers import pipeline
from textblob import TextBlob

# Rule-based category detection
def rule_based_category(message):
    message_lower = message.lower()
    if any(word in message_lower for word in ["water", "leak", "pipe", "flood"]):
        return "WATER"
    elif any(word in message_lower for word in ["energy", "air conditioning", "electricity", "lights", "ac"]):
        return "ENERGY"
    elif any(word in message_lower for word in ["recycling", "waste", "bin", "disposal"]):
        return "WASTE"
    elif any(word in message_lower for word in ["supplier", "procurement", "vendor"]):
        return "SUPPLIER"
    elif any(word in message_lower for word in ["accessible", "accessibility", "entrance", "blocked"]):
        return "ACCESSIBILITY"
    else:
        return "OTHER"

# Hugging Face DistilBERT classifier
print("Loading Hugging Face model...")
hf_classifier = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)
print("Model loaded!")

# Run Hugging Face classification
hf_results = []
print("\n--- Hugging Face Results ---")
for message in esg_messages:
    hf_output = hf_classifier(message)[0]
    sentiment = hf_output["label"]
    confidence = round(hf_output["score"], 4)
    followup = "Y" if sentiment == "NEGATIVE" else "N"
    urgency = "HIGH" if sentiment == "NEGATIVE" else "LOW"
    category = rule_based_category(message)

    result = {
        "issue_category": category,
        "urgency": urgency,
        "sentiment": sentiment,
        "followup_required": followup,
        "confidence_score": confidence
    }
    hf_results.append({"message": message, "classification": result})
    print(f"\nMessage: {message}")
    print(json.dumps(result, indent=2))

# Run TextBlob classification
textblob_results = []
print("\n--- TextBlob Results ---")
for message in esg_messages:
    polarity = TextBlob(message).sentiment.polarity
    sentiment = "NEGATIVE" if polarity < -0.1 else "POSITIVE" if polarity > 0.1 else "NEUTRAL"
    followup = "Y" if sentiment == "NEGATIVE" else "N"
    category = rule_based_category(message)

    result = {
        "issue_category": category,
        "sentiment": sentiment,
        "followup_required": followup,
        "polarity_score": round(polarity, 4)
    }
    textblob_results.append({"message": message, "classification": result})
    print(f"\nMessage: {message}")
    print(json.dumps(result, indent=2))

print("\nBaseline classification complete!")

Loading Hugging Face model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Model loaded!

--- Hugging Face Results ---

Message: There is a water leak in Building C that has been running all morning.
{
  "issue_category": "WATER",
  "urgency": "HIGH",
  "sentiment": "NEGATIVE",
  "followup_required": "Y",
  "confidence_score": 0.998
}

Message: The recycling bins are contaminated again and no one seems to be checking them.
{
  "issue_category": "WASTE",
  "urgency": "HIGH",
  "sentiment": "NEGATIVE",
  "followup_required": "Y",
  "confidence_score": 0.9996
}

Message: The air conditioning is running overnight in an empty office.
{
  "issue_category": "ENERGY",
  "urgency": "HIGH",
  "sentiment": "NEGATIVE",
  "followup_required": "Y",
  "confidence_score": 0.9995
}

Message: I want to report that one of our suppliers may not meet our sustainability policy.
{
  "issue_category": "SUPPLIER",
  "urgency": "HIGH",
  "sentiment": "NEGATIVE",
  "followup_required": "Y",
  "confidence_score": 0.9997
}

Message: The accessible entrance near the main building has been

In [16]:
# Save all results to JSON file
output = {
    "experiment": "BUS5001_A3_Q3_ESG_Message_Triage",
    "model_used": "gemini-2.5-flash",
    "baseline_models": ["distilbert-base-uncased-finetuned-sst-2-english", "TextBlob"],
    "gemini_results": gemini_results,
    "huggingface_results": hf_results,
    "textblob_results": textblob_results
}

with open("esg_triage_results.json", "w") as f:
    json.dump(output, f, indent=2)

print("Results saved to esg_triage_results.json")
print("\nFile ready for GitHub upload")

Results saved to esg_triage_results.json

File ready for GitHub upload
